# Notebook 2: Find Best Settings (Round 1 Optimization)

**For beginners.** This notebook will:
1. Try many different settings automatically
2. Find the best ones in about 1 hour

**What is optimization?** Instead of guessing the best learning rate, batch size, etc., we let the computer try 20 different combinations and pick the winner.

**Estimated time:** 1 hour on A100 GPU

**Prerequisite:** Finish `01_setup_and_train.ipynb` first.


## Step 1: Check GPU and Set Memory Limit

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.utils.env import set_gpu_memory_fraction

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    set_gpu_memory_fraction(0.8)
else:
    print("❌ No GPU!")

## Step 2: Set HuggingFace Token

In [ ]:
import os

os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
print("✅ Token set")

## Step 3: Load Config and Dataset

We use the same dataset and teacher model as before.

In [ ]:
from src.config import load_config
from src.data.dataset_loader import load_and_prepare_dataset
from src.data.preprocessing import preprocess_dataset
from src.data.tokenization import tokenize_dataset, get_tokenizer
from src.models.teacher_loader import load_teacher_model

config = load_config('../configs/default.yaml')

print("📥 Loading dataset...")
dataset = load_and_prepare_dataset(config.dataset, seed=42)
dataset = preprocess_dataset(dataset, config.dataset)

print("🧠 Loading teacher model (this takes 2-5 minutes)...")
teacher, _ = load_teacher_model(config.models['teacher'])

student_tokenizer = get_tokenizer(config.models['student'].name)
tokenized_dataset = tokenize_dataset(dataset, student_tokenizer, config.tokenization)

print("✅ All data and models loaded!")

## Step 4: Run Round 1 Optimization

This tries 20 different combinations of settings.

**What settings does it try?**
- Learning rate (how fast the model learns)
- LoRA rank (how many new parameters to train)
- Temperature (how "soft" the teacher answers are)
- Alpha/Beta (balance between copying teacher and correct answers)
- Batch size and more

**What you will see:**
```
--- Trial 0 ---
Parameters: {'learning_rate': 0.0001, 'lora_r': 16, ...}
Trial 0 completed. Validation loss: 2.15
--- Trial 1 ---
Trial 1 completed. Validation loss: 1.98
```

**Lower loss = better!** The computer remembers the best trial.


In [ ]:
from src.models.student_loader import load_student_model
from src.data.collators import DataCollatorForCausalLM
from src.models.distillation import create_distillation_trainer
from src.optimization.optuna_search import run_optuna_optimization, save_best_params
from src.optimization.search_space import suggest_parameters
from src.config import Config
import optuna
import gc

n_trials = 20  # Number of experiments to try
print(f"🔬 Starting Round 1 with {n_trials} trials...")
print("This will take about 1 hour.")
print("=" * 50)

def objective(trial: optuna.Trial) -> float:
    # The computer picks settings for this trial
    params = suggest_parameters(trial, config.optuna.search_space)
    
    print(f"\n--- Trial {trial.number} ---")
    print(f"Trying: lr={params['learning_rate']:.2e}, lora_r={params['lora_r']}, temp={params['temperature']}")
    
    # Make a copy of config with these settings
    trial_config = Config.from_dict(config.to_dict())
    trial_config.training.learning_rate = params['learning_rate']
    trial_config.training.weight_decay = params['weight_decay']
    trial_config.training.num_train_epochs = 1  # Quick trial: 1 epoch only
    trial_config.training.warmup_ratio = params['warmup_ratio']
    trial_config.training.per_device_train_batch_size = params['per_device_train_batch_size']
    trial_config.training.gradient_accumulation_steps = params['gradient_accumulation_steps']
    trial_config.lora.r = params['lora_r']
    trial_config.lora.lora_alpha = params['lora_alpha']
    trial_config.lora.lora_dropout = params['lora_dropout']
    trial_config.distillation.temperature = params['temperature']
    trial_config.distillation.alpha = params['alpha']
    trial_config.distillation.beta = params['beta']
    trial_config.tokenization.max_length = params['max_length']
    
    try:
        # Load student with these settings
        student, _ = load_student_model(
            trial_config.models['student'],
            lora_config=trial_config.lora
        )
        
        data_collator = DataCollatorForCausalLM(
            tokenizer=student_tokenizer,
            max_length=trial_config.tokenization.max_length
        )
        
        trial_dir = f"../artifacts/optuna/trial_{trial.number}"
        trainer = create_distillation_trainer(
            student_model=student,
            teacher_model=teacher,
            train_dataset=tokenized_dataset['train'],
            eval_dataset=tokenized_dataset['validation'],
            tokenizer=student_tokenizer,
            data_collator=data_collator,
            output_dir=trial_dir,
            temperature=trial_config.distillation.temperature,
            alpha=trial_config.distillation.alpha,
            beta=trial_config.distillation.beta,
            num_train_epochs=trial_config.training.num_train_epochs,
            per_device_train_batch_size=trial_config.training.per_device_train_batch_size,
            gradient_accumulation_steps=trial_config.training.gradient_accumulation_steps,
            learning_rate=trial_config.training.learning_rate,
            weight_decay=trial_config.training.weight_decay,
            warmup_ratio=trial_config.training.warmup_ratio,
            logging_steps=50,
            eval_steps=100,
            save_steps=1000,
            bf16=trial_config.hardware.mixed_precision == 'bf16',
            fp16=trial_config.hardware.mixed_precision == 'fp16',
            dataloader_num_workers=trial_config.training.dataloader_num_workers,
            gradient_checkpointing=trial_config.hardware.gradient_checkpointing,
        )
        
        trainer.train()
        metrics = trainer.evaluate()
        eval_loss = metrics.get('eval_loss', float('inf'))
        
        print(f"Trial {trial.number} done. Loss: {eval_loss:.4f}")
        
        # Clean up memory for next trial
        del student
        del trainer
        gc.collect()
        torch.cuda.empty_cache()
        
        return eval_loss
        
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return float('inf')

# Run the optimization
study = run_optuna_optimization(
    objective_func=objective,
    study_name='kd_round1_notebook',
    n_trials=n_trials,
    timeout=3600,  # 1 hour max
    direction='minimize',
    seed=42,
    show_progress_bar=True
)

print("\n" + "=" * 50)
print("🎉 ROUND 1 COMPLETE!")
print(f"Best loss: {study.best_value:.4f}")
print(f"Best settings: {study.best_params}")

## Step 5: Save the Best Settings

In [ ]:
import json

os.makedirs('../artifacts/optuna', exist_ok=True)

save_best_params(
    study,
    '../artifacts/optuna/round1_best_params.json'
)

print("✅ Best settings saved to: artifacts/optuna/round1_best_params.json")
print("\nNext step: Run notebook 03 to fine-tune these settings!")